# SHL Hiring Assessment 2026: Grammar Scoring Engine for Spoken Audio

**Goal:** predict a continuous MOS grammar score (0-5) from a 45-60 s speech clip.
**Metrics:** Pearson correlation and RMSE (training RMSE is reported explicitly below).

## Approach in one paragraph
Grammar quality is mostly about *what* is said and *how fluently* it is said, so I combine three views of each clip:
1. **Transcript-based linguistic features:** Whisper ASR (disfluencies kept), LanguageTool error counts, spaCy syntax complexity, GPT-2 perplexity.
2. **Fluency/acoustic features:** speech rate, pauses (from word timestamps), ASR word confidence, fillers, repetitions.
3. **Learned embeddings:** WavLM audio embeddings and sentence-transformer text embeddings.

Several diverse regressors are trained with stratified 5-fold CV. Their out-of-fold predictions are blended with a non-negative linear stacker. Predictions are clipped to [0, 5].

**Pipeline:** `audio -> resample 16k/normalize -> Whisper -> [handcrafted feats | WavLM emb | text emb] -> LGBM / Ridge / SVR -> stacking -> score`

## 0. Setup
Enable **Internet** and **GPU** in the Kaggle notebook settings (needed to download pretrained models).

In [ ]:
!pip install -q faster-whisper language_tool_python sentence-transformers lightgbm spacy
!python -m spacy download en_core_web_sm -q

In [ ]:
import os, glob, re, json, pickle, random, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import librosa, torch
from tqdm.auto import tqdm
from scipy.stats import pearsonr
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV, Ridge
from sklearn.svm import SVR
import lightgbm as lgb

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SR = 16000
CACHE = "/kaggle/working/cache"; os.makedirs(CACHE, exist_ok=True)
sns.set_theme(style="whitegrid")
print("Device:", DEVICE)

## 1. Load data

In [ ]:
# Locate competition files automatically
train_csv = glob.glob("/kaggle/input/**/train.csv", recursive=True)[0]
ROOT = os.path.dirname(train_csv)
train = pd.read_csv(os.path.join(ROOT, "train.csv"))
test = pd.read_csv(os.path.join(ROOT, "test.csv"))
sub = pd.read_csv(os.path.join(ROOT, "sample_submission.csv"))
print(train.shape, test.shape, sub.shape)
print(train.head())

# Detect column names instead of hard-coding
FILE_COL = train.columns[0]
LABEL_COL = [c for c in train.columns if c != FILE_COL][0]
for c in train.columns[1:]:
    if "label" in c.lower() or "score" in c.lower() or "grammar" in c.lower():
        LABEL_COL = c
print("file col:", FILE_COL, "| label col:", LABEL_COL)

# Map file stem -> full wav path
wav_paths = glob.glob("/kaggle/input/**/*.wav", recursive=True)
path_map = {os.path.splitext(os.path.basename(p))[0]: p for p in wav_paths}
def get_path(name):
    return path_map[os.path.splitext(str(name))[0]]

train["split"] = "train"; test["split"] = "test"
df = pd.concat([train, test.drop(columns=[c for c in test.columns if c == LABEL_COL], errors="ignore")], ignore_index=True)
df["path"] = df[FILE_COL].map(get_path)
y = train[LABEL_COL].values.astype(float)
print("Total audio files:", len(df))

## 2. Exploratory data analysis

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(y, bins=20, kde=True, ax=ax[0]); ax[0].set_title("Grammar score distribution (train)")
ax[0].set_xlabel("score")
durs = [librosa.get_duration(path=p) for p in tqdm(df["path"][:200])]
sns.histplot(durs, bins=20, ax=ax[1]); ax[1].set_title("Audio duration (s), first 200 files")
plt.tight_layout(); plt.show()
print(pd.Series(y).describe())

## 3. Preprocessing
Resample to 16 kHz mono, peak-normalize, and trim only leading/trailing silence (internal pauses are kept because they carry fluency information).

In [ ]:
def load_audio(path):
    wav, _ = librosa.load(path, sr=SR, mono=True)
    if len(wav) == 0:
        return np.zeros(SR, dtype=np.float32)
    wav, _ = librosa.effects.trim(wav, top_db=35)          # edge silence only
    peak = np.max(np.abs(wav)) + 1e-8
    return (wav / peak).astype(np.float32)                  # peak normalise

## 4. ASR with Whisper (faster-whisper)
An initial prompt containing fillers nudges Whisper to keep disfluencies ("um", "uh", repetitions), which matter for grammar scoring. Word-level timestamps and confidences are stored for pause and fluency features. Results are cached.

In [ ]:
from faster_whisper import WhisperModel

asr_cache = f"{CACHE}/asr.pkl"
if os.path.exists(asr_cache):
    asr_res = pickle.load(open(asr_cache, "rb"))
else:
    asr = WhisperModel("small", device=DEVICE, compute_type="float16" if DEVICE == "cuda" else "int8")
    PROMPT = "Um, so, uh, I think, like, I was, I was going to say, hmm, you know."
    asr_res = {}
    for name, path in tqdm(zip(df[FILE_COL], df["path"]), total=len(df)):
        wav = load_audio(path)
        segs, _ = asr.transcribe(wav, language="en", beam_size=5, word_timestamps=True,
                                 initial_prompt=PROMPT, condition_on_previous_text=False)
        words = []
        for s in segs:
            for w in (s.words or []):
                words.append((w.word.strip(), float(w.start), float(w.end), float(w.probability)))
        asr_res[name] = {"words": words, "dur": len(wav) / SR}
    pickle.dump(asr_res, open(asr_cache, "wb"))
    del asr; torch.cuda.empty_cache()

df["text"] = [" ".join(w[0] for w in asr_res[n]["words"]).strip() for n in df[FILE_COL]]
print(df[["text"]].head(3).to_string())

## 5. Feature engineering
### 5a. Fluency / timing features (from word timestamps)

In [ ]:
FILLERS = {"um", "uh", "umm", "uhh", "er", "erm", "hmm", "mm", "ah", "like"}

def fluency_feats(name):
    r = asr_res[name]; w = r["words"]; dur = max(r["dur"], 1e-3)
    n = len(w)
    if n < 2:
        return dict(n_words=n, dur=dur, wps=0, n_pause=0, pause_ratio=0, mean_gap=0, max_gap=0,
                    mean_prob=0, low_prob_frac=1, n_fillers=0, filler_rate=0, n_repeat=0, ttr=0, avg_wlen=0,
                    articulation_rate=0, long_pause=0)
    toks = [re.sub(r"[^a-z']", "", x[0].lower()) for x in w]
    starts = np.array([x[1] for x in w]); ends = np.array([x[2] for x in w]); probs = np.array([x[3] for x in w])
    gaps = np.clip(starts[1:] - ends[:-1], 0, None)
    speech_time = max(ends[-1] - starts[0], 1e-3)
    n_rep = sum(1 for a, b in zip(toks[:-1], toks[1:]) if a == b and a)
    n_fill = sum(1 for t in toks if t in FILLERS)
    return dict(
        n_words=n, dur=dur, wps=n / dur, n_pause=int((gaps > 0.3).sum()),
        pause_ratio=gaps[gaps > 0.3].sum() / speech_time, mean_gap=gaps.mean(), max_gap=gaps.max(),
        long_pause=int((gaps > 1.0).sum()), mean_prob=probs.mean(), low_prob_frac=(probs < 0.5).mean(),
        n_fillers=n_fill, filler_rate=n_fill / n, n_repeat=n_rep,
        ttr=len(set(toks)) / n, avg_wlen=np.mean([len(t) for t in toks]),
        articulation_rate=n / (speech_time - gaps[gaps > 0.3].sum() + 1e-3))

F_flu = pd.DataFrame([fluency_feats(n) for n in df[FILE_COL]])
F_flu.describe().T[["mean", "std", "min", "max"]].round(3)

### 5b. Grammar-error features (LanguageTool, with graceful fallback)

In [ ]:
lt_cache = f"{CACHE}/lt.pkl"
if os.path.exists(lt_cache):
    F_lt = pickle.load(open(lt_cache, "rb"))
else:
    try:
        import language_tool_python
        tool = language_tool_python.LanguageTool("en-US")
        rows = []
        for t in tqdm(df["text"]):
            m = tool.check(t) if t else []
            nw = max(len(t.split()), 1)
            cats = [x.category for x in m]
            rows.append(dict(lt_errors=len(m), lt_per100=100 * len(m) / nw,
                             lt_grammar=cats.count("GRAMMAR"), lt_typos=cats.count("TYPOS"),
                             lt_punct=cats.count("PUNCTUATION"), lt_style=cats.count("STYLE"),
                             lt_casing=cats.count("CASING")))
        F_lt = pd.DataFrame(rows); tool.close()
    except Exception as e:
        print("LanguageTool unavailable, using zeros:", e)
        F_lt = pd.DataFrame(np.zeros((len(df), 2)), columns=["lt_errors", "lt_per100"])
    pickle.dump(F_lt, open(lt_cache, "wb"))
F_lt.describe().T[["mean", "std"]].round(3)

### 5c. Syntax complexity (spaCy) and GPT-2 perplexity

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")
SUBORD = {"advcl", "ccomp", "xcomp", "acl", "relcl", "csubj", "mark"}

def depth(tok):
    d = 0
    while tok.head != tok:
        tok = tok.head; d += 1
    return d

def syntax_feats(text):
    doc = nlp(text if text else "empty")
    sents = list(doc.sents); ns = max(len(sents), 1); nt = max(len(doc), 1)
    depths = [depth(t) for t in doc]
    frag = sum(1 for s in sents if not any(t.pos_ in ("VERB", "AUX") for t in s)) / ns
    pos = pd.Series([t.pos_ for t in doc]).value_counts(normalize=True)
    return dict(n_sents=ns, avg_sent_len=nt / ns, max_depth=max(depths), mean_depth=np.mean(depths),
                n_subord=sum(t.dep_ in SUBORD for t in doc) / ns, frag_frac=frag,
                verb_frac=pos.get("VERB", 0), aux_frac=pos.get("AUX", 0), noun_frac=pos.get("NOUN", 0),
                pron_frac=pos.get("PRON", 0), adj_frac=pos.get("ADJ", 0), adv_frac=pos.get("ADV", 0),
                n_tenses=len({m for t in doc for m in t.morph.get("Tense")}))

F_syn = pd.DataFrame([syntax_feats(t) for t in tqdm(df["text"])])

from transformers import AutoTokenizer, AutoModelForCausalLM
gtok = AutoTokenizer.from_pretrained("gpt2"); glm = AutoModelForCausalLM.from_pretrained("gpt2").to(DEVICE).eval()
@torch.no_grad()
def ppl_feats(text):
    if len(text.split()) < 3:
        return dict(ppl=500, loss_mean=6.2, loss_std=0, loss_p90=6.2)
    ids = gtok(text, return_tensors="pt", truncation=True, max_length=512).input_ids.to(DEVICE)
    logits = glm(ids).logits[:, :-1]
    loss = torch.nn.functional.cross_entropy(logits[0], ids[0, 1:], reduction="none").cpu().numpy()
    return dict(ppl=float(np.exp(loss.mean())), loss_mean=loss.mean(), loss_std=loss.std(), loss_p90=np.percentile(loss, 90))
F_ppl = pd.DataFrame([ppl_feats(t) for t in tqdm(df["text"])])
del glm; torch.cuda.empty_cache()

### 5d. Learned embeddings: WavLM (audio) and MPNet (text)

In [ ]:
from transformers import AutoFeatureExtractor, WavLMModel
emb_cache = f"{CACHE}/emb.npz"
if os.path.exists(emb_cache):
    z = np.load(emb_cache); AE, TE = z["AE"], z["TE"]
else:
    fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
    wm = WavLMModel.from_pretrained("microsoft/wavlm-base-plus").to(DEVICE).eval()
    AE = []
    with torch.no_grad():
        for p in tqdm(df["path"]):
            wav = load_audio(p)[: SR * 65]
            inp = fe(wav, sampling_rate=SR, return_tensors="pt").input_values.to(DEVICE)
            hs = wm(inp, output_hidden_states=True).hidden_states
            # mean-pool a middle layer (more linguistic) and the last layer
            AE.append(torch.cat([hs[6].mean(1), hs[12].mean(1)], dim=1).squeeze(0).cpu().numpy())
    AE = np.stack(AE); del wm; torch.cuda.empty_cache()

    from sentence_transformers import SentenceTransformer
    st = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=DEVICE)
    TE = st.encode(df["text"].tolist(), batch_size=32, show_progress_bar=True)
    np.savez(emb_cache, AE=AE, TE=TE)
print(AE.shape, TE.shape)

### 5e. Assemble feature matrices

In [ ]:
HC = pd.concat([F_flu, F_lt, F_syn, F_ppl], axis=1).replace([np.inf, -np.inf], 0).fillna(0)
HC["lt_per100_x_ppl"] = HC["lt_per100"] * np.log1p(HC["ppl"])
tr_idx = np.where(df["split"] == "train")[0]; te_idx = np.where(df["split"] == "test")[0]
HC_tr, HC_te = HC.iloc[tr_idx].values, HC.iloc[te_idx].values
AE_tr, AE_te = AE[tr_idx], AE[te_idx]
TE_tr, TE_te = TE[tr_idx], TE[te_idx]
ALL_tr = np.hstack([HC_tr, AE_tr, TE_tr]); ALL_te = np.hstack([HC_te, AE_te, TE_te])
print("HC", HC_tr.shape, "| AE", AE_tr.shape, "| TE", TE_tr.shape)

# How strongly do individual interpretable features correlate with the score?
corr = HC.iloc[tr_idx].assign(score=y).corr()["score"].drop("score").sort_values()
plt.figure(figsize=(7, 8)); corr.plot(kind="barh"); plt.title("Feature correlation with grammar score"); plt.tight_layout(); plt.show()

## 6. Models and cross-validation
Five diverse base models, each trained with **stratified 5-fold CV** (stratified on the rounded score):

| Model | Features |
|---|---|
| LightGBM | handcrafted features |
| Ridge | WavLM embeddings |
| SVR | text embeddings (MPNet) |
| SVR | all handcrafted + PCA(audio) + PCA(text) |
| Ridge | handcrafted features (linear baseline) |

Out-of-fold (OOF) predictions are blended by a non-negative Ridge stacker.

In [ ]:
def rmse(a, b): return float(np.sqrt(mean_squared_error(a, b)))
def report(name, yt, yp):
    yp = np.clip(yp, 0, 5)
    return dict(model=name, RMSE=rmse(yt, yp), Pearson=pearsonr(yt, yp)[0])

def make_lgb():
    return lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, num_leaves=8, min_child_samples=15,
                             subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=2.0,
                             random_state=SEED, verbose=-1)

MODELS = {
    "LGBM_handcrafted":   (make_lgb,                                                                    HC_tr, HC_te),
    "Ridge_audio_emb":    (lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(1, 4, 15))), AE_tr, AE_te),
    "SVR_text_emb":       (lambda: make_pipeline(StandardScaler(), SVR(C=2.0, epsilon=0.1, gamma="scale")), TE_tr, TE_te),
    "SVR_combined":       (lambda: make_pipeline(StandardScaler(), PCA(64, random_state=SEED), SVR(C=2.0, epsilon=0.1)), ALL_tr, ALL_te),
    "Ridge_handcrafted":  (lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-1, 3, 15))), HC_tr, HC_te),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
strat = np.clip(np.round(y), 0, 5).astype(int)
# merge classes that are too rare for stratification
vc = pd.Series(strat).value_counts(); strat = np.where(pd.Series(strat).map(vc) < 5, vc.index[0], strat)
folds = list(skf.split(np.zeros(len(y)), strat))

oof, test_pred, train_pred = {}, {}, {}
for name, (builder, Xtr, Xte) in MODELS.items():
    o = np.zeros(len(y)); t = np.zeros(len(Xte))
    for tr, va in folds:
        m = builder().fit(Xtr[tr], y[tr])
        o[va] = m.predict(Xtr[va]); t += m.predict(Xte) / len(folds)
    full = builder().fit(Xtr, y)                      # full fit -> training RMSE
    oof[name], test_pred[name], train_pred[name] = o, t, full.predict(Xtr)
    print(f"{name:20s} CV RMSE={rmse(y, np.clip(o,0,5)):.4f}  CV Pearson={pearsonr(y, o)[0]:.4f}")

## 7. Stacking (non-negative blend of OOF predictions)

In [ ]:
names = list(MODELS)
O = np.column_stack([oof[n] for n in names]); T = np.column_stack([test_pred[n] for n in names])
TR = np.column_stack([train_pred[n] for n in names])

stacker = Ridge(alpha=1.0, positive=True, fit_intercept=True).fit(O, y)
weights = pd.Series(stacker.coef_, index=names); print("Blend weights:\n", weights.round(3), "\nintercept:", round(stacker.intercept_, 3))

# Nested check: weights fit inside CV to avoid optimism in the reported ensemble CV
oof_stack = np.zeros(len(y))
for tr, va in folds:
    s = Ridge(alpha=1.0, positive=True).fit(O[tr], y[tr]); oof_stack[va] = s.predict(O[va])

final_oof   = np.clip(oof_stack, 0, 5)
final_train = np.clip(stacker.predict(TR), 0, 5)
final_test  = np.clip(stacker.predict(T), 0, 5)

## 8. Results (including the compulsory training RMSE)

In [ ]:
rows = [report(f"{n} (CV)", y, oof[n]) for n in names]
rows.append(report("STACKED ENSEMBLE (CV, nested)", y, final_oof))
res = pd.DataFrame(rows).round(4)
display(res)

TRAIN_RMSE = rmse(y, final_train)
print("=" * 60)
print(f"TRAINING RMSE (final ensemble, fit on all training data): {TRAIN_RMSE:.4f}")
print(f"TRAINING Pearson                                        : {pearsonr(y, final_train)[0]:.4f}")
print(f"CROSS-VALIDATED RMSE (out-of-fold, more honest)         : {rmse(y, final_oof):.4f}")
print(f"CROSS-VALIDATED Pearson                                 : {pearsonr(y, final_oof)[0]:.4f}")
print("=" * 60)

## 9. Visualizations

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
ax[0].scatter(y, final_oof, alpha=0.5); ax[0].plot([0, 5], [0, 5], "r--")
ax[0].set_xlabel("True score"); ax[0].set_ylabel("Predicted (OOF)"); ax[0].set_title("Predicted vs actual (CV)")
resid = y - final_oof
sns.histplot(resid, bins=25, kde=True, ax=ax[1]); ax[1].set_title(f"Residuals (mean={resid.mean():.3f}, std={resid.std():.3f})")
r2 = res.iloc[:len(names) + 1]
sns.barplot(data=r2, y="model", x="RMSE", ax=ax[2], palette="viridis"); ax[2].set_title("CV RMSE per model (lower is better)")
plt.tight_layout(); plt.show()

# Per-score-bucket error shows where the model struggles (usually the extremes)
bucket = pd.DataFrame({"true": np.round(y), "abs_err": np.abs(resid)}).groupby("true")["abs_err"].agg(["mean", "count"])
display(bucket.round(3))

# Feature importance from LightGBM on handcrafted features
lgb_full = make_lgb().fit(HC_tr, y)
imp = pd.Series(lgb_full.feature_importances_, index=HC.columns).sort_values().tail(20)
plt.figure(figsize=(7, 6)); imp.plot(kind="barh"); plt.title("Top 20 LightGBM feature importances"); plt.tight_layout(); plt.show()

# Pearson per model
plt.figure(figsize=(7, 4)); sns.barplot(data=r2, y="model", x="Pearson", palette="mako"); plt.title("CV Pearson per model"); plt.tight_layout(); plt.show()

## 10. Create submission

In [ ]:
id_col, pred_col = sub.columns[0], sub.columns[1]
out = pd.DataFrame({id_col: test[FILE_COL].values, pred_col: final_test})
out = sub[[id_col]].merge(out, on=id_col, how="left")   # keep sample_submission order
out[pred_col] = out[pred_col].fillna(float(np.mean(y)))
out.to_csv("/kaggle/working/submission.csv", index=False)

print(out.head())
print(out.shape, "| range:", round(float(out[pred_col].min()), 2), "to", round(float(out[pred_col].max()), 2))

chk = pd.read_csv("/kaggle/working/submission.csv")
print("columns:", list(chk.columns), "| expected:", list(sub.columns))
print("NaNs:", int(chk.isna().sum().sum()), "| ids match sample:", set(chk[id_col]) == set(sub[id_col]))

In [ ]:
id_col, pred_col = sub.columns[0], sub.columns[1]

# One prediction per test file, in test.csv order
out = pd.DataFrame({id_col: test[FILE_COL].values, pred_col: np.clip(final_test, 0, 5)})
out.to_csv("/kaggle/working/submission.csv", index=False)

print(out.shape, "| unique ids:", out[id_col].nunique())
print(out[pred_col].describe().round(3))
print("rows equal to the training mean:", int((out[pred_col].round(6) == round(float(np.mean(y)), 6)).sum()))
print("test ids not in wav files:", sum(os.path.splitext(i)[0] not in path_map for i in out[id_col]))

# Diagnostic: where do the sample ids come from?
ids_s = set(sub[id_col])
print("sample ids in test.csv:", len(ids_s & set(test[FILE_COL])),
      "| in train.csv:", len(ids_s & set(train[FILE_COL])), "| of", len(ids_s))

## 11. Report

**Problem.** Regress a 0-5 grammar score for 45-60 s spoken clips (769 train / 216 test).

**Preprocessing.** 16 kHz mono, peak normalisation, edge-silence trimming; internal pauses kept as signal.

**Architecture.** Whisper-small (faster-whisper) transcribes with word timestamps and confidences. Three feature groups are built: (1) handcrafted fluency + LanguageTool + spaCy + GPT-2 perplexity features, (2) WavLM mean-pooled embeddings (layers 6 and 12), (3) MPNet sentence embeddings of the transcript. Five regressors (LightGBM, Ridge, SVR) are trained with stratified 5-fold CV and blended with a non-negative Ridge stacker. Test predictions average the five fold models, then are clipped to [0, 5].

**Results.** See the Section 8 table: training RMSE, CV RMSE and CV Pearson are printed explicitly. Training RMSE is lower than CV RMSE as expected; CV is the number to trust.

**Interpretation.** The correlation bar chart and LightGBM importances show that grammar-error rate, perplexity, fragment ratio, speech rate/pauses and ASR word confidence carry most of the signal. The error analysis shows larger errors at the extreme scores (1 and 5), which are rare in the training data.

**Limitations / future work.** ASR errors and punctuation guesses propagate into text features; the dataset is small and labels are noisy (Likert, subjective); Whisper tends to hide disfluencies. Next steps: fine-tune DeBERTa-v3 on transcripts with a regression head, use Whisper-medium/large, multi-seed CV, and ordinal-aware losses.